In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import os
from google.colab import drive, userdata
import subprocess

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Setup GitHub integration and pull latest changes

# Configuration for Git
GIT_REPO_PATH = "/content/drive/MyDrive/master_thesis_FM/your_repo_folder" # UPDATE THIS PATH
BRANCH_NAME = "main" # UPDATE THIS IF NEEDED

# Switch to the repository directory
os.chdir(GIT_REPO_PATH)

# Retrieve Personal Access Token from Colab Secrets
try:
    pat = userdata.get('GITHUB_PAT')
    # Configure git to use credentials if necessary (optional depending on repo visibility)
    # subprocess.run(["git", "config", "--global", "credential.helper", "store"])
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_PAT not found in Colab Secrets. Pulling may fail if the repo is private.")

# Fetch and pull from the specific branch
print(f"Fetching and pulling from branch: {BRANCH_NAME}...")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", BRANCH_NAME], check=True)
subprocess.run(["git", "pull", "origin", BRANCH_NAME], check=True)
print("Git setup complete.")

In [ ]:
# Install necessary dependencies
#!pip install rasterio h5py psutil tqdm matplotlib pandas
# AnySat requirements (assuming requirements.txt is in the repo feature_extraction folder)
# !pip install -r /content/drive/MyDrive/master_thesis_FM/models/1_AnySat/feature_extraction/requirements.txt

In [3]:
from dataclasses import dataclass
import torch

@dataclass(frozen=True)
class Config:
    # Reproducibility
    SEED: int = 42

    # Hardware & DataLoader
    NUM_WORKERS: int = 4
    BATCH_SIZE: int = 8
    MATH_BATCH_SIZE: int = 8

    # Optimizer & Training
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 0.05
    EPOCHS: int = 100
    PATIENCE: int = 10

    # AnySat Parameters
    S1_DATE: int = 196
    PATCH_SIZE: int = 40
    CHANNELS: int = 3

    # Drive Archive Paths (UPDATE THESE PATHS)
    ARCHIVE_TRAIN_SAR: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz"
    ARCHIVE_VAL_SAR: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/val_20%_random_out_of_orig_train_partition.tar.gz"
    ARCHIVE_TEST_SAR: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/orig_test_partition.tar.gz"
    ARCHIVE_TRAIN_LBL: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz"
    ARCHIVE_VAL_LBL: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/val_20%_random_out_of_orig_train_partition.tar.gz"
    ARCHIVE_TEST_LBL: str = "/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/test.tar.gz"


    # Local Storage Extraction Paths (Fast I/O)
    LOCAL_DATA_DIR: str = "/content/local_data"

    # Drive Output Paths
    EXP_NAME: str = f"AS_LP_OTF_RANDOM10%_train_v2_patSize{PATCH_SIZE}_BS{MATH_BATCH_SIZE}_LR{str(LEARNING_RATE).split('.')[1]}_WD{str(WEIGHT_DECAY).split('.')[1]}_EP{EPOCHS}_PAT{PATIENCE}_SEED{SEED}"
    OUTPUT_DIR: str = f"/content/drive/MyDrive/master_thesis_FM/experiments/AnySat/linear_probing/dense_features/{EXP_NAME}"

CONFIG = Config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import random
import numpy as np
import os

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def clear_memory():
    import gc
    gc.collect()
    torch.cuda.empty_cache()

set_seed(CONFIG.SEED)
os.makedirs(CONFIG.OUTPUT_DIR, exist_ok=True)

In [5]:
import subprocess
import shutil

def process_archive(archive_path, target_subfolder):
    """Copies an archive to local storage, extracts it, and deletes the archive."""
    if not os.path.exists(archive_path):
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    extract_path = os.path.join(CONFIG.LOCAL_DATA_DIR, target_subfolder)
    os.makedirs(extract_path, exist_ok=True)

    local_archive_path = os.path.join(CONFIG.LOCAL_DATA_DIR, os.path.basename(archive_path))

    print(f"Copying {archive_path} to local storage...")
    shutil.copy2(archive_path, local_archive_path)

    print(f"Extracting to {extract_path}...")
    subprocess.run(["tar", "-xzf", local_archive_path, "-C", extract_path], check=True)

    print(f"Deleting local archive {local_archive_path}...")
    os.remove(local_archive_path)
    print(f"Finished processing {target_subfolder}.\n")

# Process archives sequentially to minimize local storage spikes
print("Starting sequential data extraction...")
archives = {
    "train/sar": CONFIG.ARCHIVE_TRAIN_SAR, "train/label": CONFIG.ARCHIVE_TRAIN_LBL,
    "val/sar": CONFIG.ARCHIVE_VAL_SAR, "val/label": CONFIG.ARCHIVE_VAL_LBL,
    "test/sar": CONFIG.ARCHIVE_TEST_SAR, "test/label": CONFIG.ARCHIVE_TEST_LBL
}

for folder, path in archives.items():
    # To prevent re-extraction upon notebook restart
    if not os.path.exists(os.path.join(CONFIG.LOCAL_DATA_DIR, folder)):
        process_archive(path, folder)
    else:
        print(f"Data already extracted in {folder}, skipping.")

Starting sequential data extraction...
Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tensors_ASC_3channels_VV_VH_ratio/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz to local storage...
Extracting to /content/local_data/train/sar...
Deleting local archive /content/local_data/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz...
Finished processing train/sar.

Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/labels/full_dataset/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz to local storage...
Extracting to /content/local_data/train/label...
Deleting local archive /content/local_data/train_10%_random_out_of_80%_of_orig_train_partition.tar.gz...
Finished processing train/label.

Copying /content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/packed_compressed_data/tar_gz_archives/standardized_s1_july_pt_tenso

In [6]:
import glob
import re
import rasterio
from torch.utils.data import Dataset, DataLoader

class BioMasstersDataset(Dataset):
    def __init__(self, sar_dir, label_dir):
        self.sar_paths = sorted(glob.glob(os.path.join(sar_dir, "**", "*.pt"), recursive=True))
        self.label_paths = sorted(glob.glob(os.path.join(label_dir, "**", "*.tif"), recursive=True))

        # Mapping using the 8-character ID
        self.sar_map = {}
        sar_pattern = re.compile(r'_([A-Za-z0-9]{8})_S1_10\.pt$')
        for path in self.sar_paths:
            match = sar_pattern.search(path)
            if match:
                self.sar_map[match.group(1)] = path

        self.label_map = {}
        lbl_pattern = re.compile(r'([A-Za-z0-9]{8})_agbm\.tif$')
        for path in self.label_paths:
            match = lbl_pattern.search(path)
            if match:
                self.label_map[match.group(1)] = path

        # Intersection of keys ensures complete pairs
        self.valid_ids = sorted(list(set(self.sar_map.keys()) & set(self.label_map.keys())))
        print(f"Found {len(self.valid_ids)} matched files in directory {sar_dir}")

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        file_id = self.valid_ids[idx]

        # Load SAR tensor: shape [C, H, W]
        sar_tensor = torch.load(self.sar_map[file_id], map_location='cpu').float()

        # Load Label using rasterio: shape [H, W]
        with rasterio.open(self.label_map[file_id]) as src:
            label_array = src.read(1)

        # Replace NaNs or invalid values if necessary
        label_tensor = torch.from_numpy(label_array).float()

        return sar_tensor, label_tensor, file_id

# Instantiate datasets and dataloaders
train_dataset = BioMasstersDataset(os.path.join(CONFIG.LOCAL_DATA_DIR, "train/sar"), os.path.join(CONFIG.LOCAL_DATA_DIR, "train/label"))
val_dataset = BioMasstersDataset(os.path.join(CONFIG.LOCAL_DATA_DIR, "val/sar"), os.path.join(CONFIG.LOCAL_DATA_DIR, "val/label"))

train_loader = DataLoader(train_dataset, batch_size=CONFIG.BATCH_SIZE, shuffle=True, num_workers=CONFIG.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG.BATCH_SIZE, shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=True)

Found 696 matched files in directory /content/local_data/train/sar
Found 1738 matched files in directory /content/local_data/val/sar


In [7]:
import torch.nn as nn

class BiomassEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pretrained model from Torch Hub
        self.anysat = torch.hub.load('gastruc/anysat', 'anysat', pretrained=True, force_reload=False, flash_attn=False)#, trust_repo='check')

        # Freeze AnySat parameters
        for param in self.anysat.parameters():
            param.requires_grad = False

        # Linear Probe equivalent using 1x1 Conv
        # Input channels: 1536 (from AnySat dense features)
        # Output channels: 1 (AGB value per pixel)
        self.probe = nn.Conv2d(in_channels=1536, out_channels=1, kernel_size=1)

    def forward(self, sar_batch, dates_batch):
        # Package data as required by AnySat
        data = {
            "s1": sar_batch,        # Shape: [B, 1, 3, 256, 256]
            "s1_dates": dates_batch # Shape: [B]
        }

        with torch.no_grad():
            # Feature extraction on the fly
            features = self.anysat(data, patch_size=CONFIG.PATCH_SIZE, output='dense', output_modality='s1')
            # features shape: [B, 256, 256, 1536]

        # Permute for PyTorch Conv2d input standard: [B, C, H, W]
        features = features.permute(0, 3, 1, 2)

        # Probe and collapse the channel dimension
        preds = self.probe(features).squeeze(1) # Final shape: [B, 256, 256]
        return preds

model = BiomassEstimator().to(DEVICE)

The repository gastruc_anysat does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/gastruc/anysat/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://huggingface.co/g-astruc/AnySat/resolve/main/models/AnySat.pth" to /root/.cache/torch/hub/checkpoints/AnySat.pth


100%|██████████| 480M/480M [00:01<00:00, 268MB/s]


In [8]:
class StreamingMetrics:
    """Calculates evaluation metrics globally without storing all pixels in RAM."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum_squared_error = 0.0
        self.sum_absolute_error = 0.0
        self.sum_y = 0.0
        self.sum_y_sq = 0.0
        self.count = 0

    def update(self, preds, targets):
        # Flatten tensors for calculation
        preds = preds.detach().reshape(-1)
        targets = targets.detach().reshape(-1)

        # Mask out any NaN labels if they exist in your dataset
        valid_mask = ~torch.isnan(targets)
        preds = preds[valid_mask]
        targets = targets[valid_mask]

        n = targets.numel()
        if n == 0:
            return

        self.count += n
        self.sum_squared_error += torch.sum((preds - targets) ** 2).item()
        self.sum_absolute_error += torch.sum(torch.abs(preds - targets)).item()
        self.sum_y += torch.sum(targets).item()
        self.sum_y_sq += torch.sum(targets ** 2).item()

    def compute(self):
        if self.count == 0:
            return {"MSE": 0, "RMSE": 0, "MAE": 0, "R2": 0}

        mse = self.sum_squared_error / self.count
        rmse = np.sqrt(mse)
        mae = self.sum_absolute_error / self.count

        # R2 calculation: 1 - (SSR / SST)
        # SST = sum(y^2) - (sum(y)^2 / n)
        sst = self.sum_y_sq - ((self.sum_y ** 2) / self.count)

        if sst == 0:
            r2 = 0.0 # Prevent division by zero if all targets are identical
        else:
            r2 = 1.0 - (self.sum_squared_error / sst)

        return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

# Setup optimizer and loss
optimizer = optim.Adam(model.probe.parameters(), lr=CONFIG.LEARNING_RATE, weight_decay=CONFIG.WEIGHT_DECAY)
criterion = nn.MSELoss()
scaler = GradScaler()

# Checkpoint paths
checkpoint_path = os.path.join(CONFIG.OUTPUT_DIR, "checkpoint_latest.pth")
best_model_path = os.path.join(CONFIG.OUTPUT_DIR, "best_model.pth")
history_path = os.path.join(CONFIG.OUTPUT_DIR, "training_history.csv")

# Training State
start_epoch = 0
best_val_loss = float('inf')
epochs_without_improvement = 0
history = []

# Resume logic
if os.path.exists(checkpoint_path):
    print(f"Resuming training from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    if os.path.exists(history_path):
        history = pd.read_csv(history_path).to_dict('records')

metrics_tracker = StreamingMetrics()

for epoch in range(start_epoch, CONFIG.EPOCHS):
    model.train()
    train_loss = 0.0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG.EPOCHS} [Train]")
    for sar, label, _ in train_pbar:
        sar, label = sar.to(DEVICE), label.to(DEVICE)

        # AnySat expects an unsqueezed channel dimension for time series (Batch, Time, Channels, H, W)
        sar = sar.unsqueeze(1)
        dates = torch.full((sar.shape[0],), CONFIG.S1_DATE, dtype=torch.long).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            preds = model(sar, dates)
            loss = criterion(preds, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_pbar.set_postfix({"Loss": loss.item()})

    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    metrics_tracker.reset()

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG.EPOCHS} [Val]")
    with torch.no_grad():
        for sar, label, _ in val_pbar:
            sar, label = sar.to(DEVICE), label.to(DEVICE)
            sar = sar.unsqueeze(1)
            dates = torch.full((sar.shape[0],), CONFIG.S1_DATE, dtype=torch.long).to(DEVICE)

            with autocast():
                preds = model(sar, dates)

            metrics_tracker.update(preds, label)

    val_metrics = metrics_tracker.compute()

    # Logging
    epoch_stats = {
        "Epoch": epoch + 1,
        "Train_MSE": avg_train_loss,
        "Val_MSE": val_metrics["MSE"],
        "Val_RMSE": val_metrics["RMSE"],
        "Val_MAE": val_metrics["MAE"],
        "Val_R2": val_metrics["R2"]
    }
    history.append(epoch_stats)
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(f"Epoch {epoch+1} Metrics | Train MSE: {avg_train_loss:.4f} | Val MSE: {val_metrics['MSE']:.4f} | Val R2: {val_metrics['R2']:.4f}")

    # Save Latest Checkpoint
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_val_loss': best_val_loss
    }, checkpoint_path)

    # Early Stopping & Best Model Save
    if val_metrics["MSE"] < best_val_loss:
        best_val_loss = val_metrics["MSE"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
        print("-> Best model saved.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG.PATIENCE:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    clear_memory()

/tmp/ipykernel_500/1089993207.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/100 [Train]:   0%|          | 0/87 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

Epoch 1/100 [Val]:   0%|          | 0/218 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

...

Epoch 14 Metrics | Train MSE: 5991.3429 | Val MSE: 5662.0887 | Val R2: -0.0778
-> Best model saved.


Epoch 15/100 [Train]:   0%|          | 0/87 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

Epoch 15/100 [Val]:   0%|          | 0/218 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset =

In [ ]:
"""
import pandas as pd
from tqdm.notebook import tqdm
import torch.optim as optim
import torch

# Configuration update for Gradient Accumulation
# Assuming CONFIG.BATCH_SIZE is now set to 2
ACCUMULATION_STEPS = 4  # 2 (Batch) x 4 (Steps) = 8 Effective Batch Size

# Setup optimizer and loss
optimizer = optim.Adam(model.probe.parameters(), lr=CONFIG.LEARNING_RATE, weight_decay=CONFIG.WEIGHT_DECAY)
criterion = nn.MSELoss()

# Modern AMP Syntax
scaler = torch.amp.GradScaler('cuda')

# Checkpoint paths
checkpoint_path = os.path.join(CONFIG.OUTPUT_DIR, "checkpoint_latest.pth")
best_model_path = os.path.join(CONFIG.OUTPUT_DIR, "best_model.pth")
history_path = os.path.join(CONFIG.OUTPUT_DIR, "training_history.csv")

# Training State
start_epoch = 0
best_val_loss = float('inf')
epochs_without_improvement = 0
history = []

# Resume logic
if os.path.exists(checkpoint_path):
    print(f"Resuming training from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    if os.path.exists(history_path):
        history = pd.read_csv(history_path).to_dict('records')

metrics_tracker = StreamingMetrics()

for epoch in range(start_epoch, CONFIG.EPOCHS):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True) # Zero gradients at the start of the epoch

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG.EPOCHS} [Train]")

    for i, (sar, label, _) in enumerate(train_pbar):
        sar, label = sar.to(DEVICE), label.to(DEVICE)

        sar = sar.unsqueeze(1)
        dates = torch.full((sar.shape[0],), CONFIG.S1_DATE, dtype=torch.long).to(DEVICE)

        # Modern AMP Syntax
        with torch.amp.autocast('cuda'):
            preds = model(sar, dates)
            # Scale the loss to account for accumulation
            loss = criterion(preds, label) / ACCUMULATION_STEPS

        # Backward pass accumulates gradients
        scaler.scale(loss).backward()

        # Only step the optimizer after ACCUMULATION_STEPS batches
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # Multiply back by accumulation steps for accurate loss reporting
        train_loss += (loss.item() * ACCUMULATION_STEPS)
        train_pbar.set_postfix({"Loss": (loss.item() * ACCUMULATION_STEPS)})

    avg_train_loss = train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    metrics_tracker.reset()

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG.EPOCHS} [Val]")
    with torch.no_grad():
        for sar, label, _ in val_pbar:
            sar, label = sar.to(DEVICE), label.to(DEVICE)
            sar = sar.unsqueeze(1)
            dates = torch.full((sar.shape[0],), CONFIG.S1_DATE, dtype=torch.long).to(DEVICE)

            with torch.amp.autocast('cuda'):
                preds = model(sar, dates)

            metrics_tracker.update(preds, label)

    val_metrics = metrics_tracker.compute()

    # Logging
    epoch_stats = {
        "Epoch": epoch + 1,
        "Train_MSE": avg_train_loss,
        "Val_MSE": val_metrics["MSE"],
        "Val_RMSE": val_metrics["RMSE"],
        "Val_MAE": val_metrics["MAE"],
        "Val_R2": val_metrics["R2"]
    }
    history.append(epoch_stats)
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(f"Epoch {epoch+1} Metrics | Train MSE: {avg_train_loss:.4f} | Val MSE: {val_metrics['MSE']:.4f} | Val R2: {val_metrics['R2']:.4f}")

    # Checkpointing logic remains the same...
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_val_loss': best_val_loss
    }, checkpoint_path)

    if val_metrics["MSE"] < best_val_loss:
        best_val_loss = val_metrics["MSE"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
        print("-> Best model saved.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG.PATIENCE:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

    clear_memory()
"""

Epoch 1/100 [Train]:   0%|          | 0/348 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves():
    df = pd.read_csv(history_path)
    plt.figure(figsize=(10, 6))
    plt.plot(df['Epoch'], df['Train_MSE'], label='Train MSE', marker='o')
    plt.plot(df['Epoch'], df['Val_MSE'], label='Validation MSE', marker='s')

    plt.title('Training and Validation MSE over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Squared Error')
    plt.grid(True)
    plt.legend()

    plot_path = os.path.join(CONFIG.OUTPUT_DIR, "learning_curves.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Plot saved to {plot_path}")

plot_learning_curves()

In [ ]:
# Create Test Dataset and Loader
test_dataset = BioMasstersDataset(os.path.join(CONFIG.LOCAL_DATA_DIR, "test/sar"), os.path.join(CONFIG.LOCAL_DATA_DIR, "test/label"))
test_loader = DataLoader(test_dataset, batch_size=CONFIG.BATCH_SIZE, shuffle=False, num_workers=CONFIG.NUM_WORKERS)

# Load best model for isolated execution safety
inference_model = BiomassEstimator().to(DEVICE)
inference_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
inference_model.eval()

test_metrics_tracker = StreamingMetrics()

print("Evaluating on Test Set...")
with torch.no_grad():
    for sar, label, _ in tqdm(test_loader, desc="Testing"):
        sar, label = sar.to(DEVICE), label.to(DEVICE)
        sar = sar.unsqueeze(1)
        dates = torch.full((sar.shape[0],), CONFIG.S1_DATE, dtype=torch.long).to(DEVICE)

        with autocast():
            preds = inference_model(sar, dates)

        test_metrics_tracker.update(preds, label)

test_results = test_metrics_tracker.compute()

# Save final results to txt
final_report_path = os.path.join(CONFIG.OUTPUT_DIR, "final_test_metrics.txt")
with open(final_report_path, "w") as f:
    f.write("=== Test Set Evaluation ===\n")
    for metric, value in test_results.items():
        f.write(f"{metric}: {value:.4f}\n")
    f.write("\n=== Best Model Info ===\n")

    history_df = pd.read_csv(history_path)
    best_epoch_row = history_df.loc[history_df['Val_MSE'].idxmin()]
    f.write(best_epoch_row.to_string())

print(f"Test evaluation completed. Results saved to {final_report_path}")

In [ ]:
def visualize_predictions(num_samples=4):
    inference_model.eval()
    samples_shown = 0

    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 5 * num_samples))
    plt.subplots_adjust(hspace=0.3)

    # Force axes to be 2D array even if num_samples is 1
    if num_samples == 1: axes = axes.reshape(1, 2)

    with torch.no_grad():
        for sar, label, file_ids in test_loader:
            sar, label = sar.to(DEVICE), label.to(DEVICE)
            sar = sar.unsqueeze(1)
            dates = torch.full((sar.shape[0],), CONFIG.S1_DATE, dtype=torch.long).to(DEVICE)

            with autocast():
                preds = inference_model(sar, dates)

            for i in range(sar.shape[0]):
                if samples_shown >= num_samples:
                    break

                target_img = label[i].cpu().numpy()
                pred_img = preds[i].cpu().numpy()
                img_id = file_ids[i]

                # Plot Target
                im1 = axes[samples_shown, 0].imshow(target_img, cmap='viridis')
                axes[samples_shown, 0].set_title(f"Target AGB - ID: {img_id}")
                plt.colorbar(im1, ax=axes[samples_shown, 0], fraction=0.046, pad=0.04)

                # Plot Prediction
                im2 = axes[samples_shown, 1].imshow(pred_img, cmap='viridis')
                axes[samples_shown, 1].set_title(f"Predicted AGB - ID: {img_id}")
                plt.colorbar(im2, ax=axes[samples_shown, 1], fraction=0.046, pad=0.04)

                samples_shown += 1

            if samples_shown >= num_samples:
                break

    viz_path = os.path.join(CONFIG.OUTPUT_DIR, "prediction_visualizations.png")
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Visualizations saved to {viz_path}")

visualize_predictions(num_samples=4)

In [ ]:
# Push all pipeline outputs to the repository
print("Flushing data to Google Drive...")
drive.flush_and_unmount() # Ensures all data is physically written to Drive before committing
drive.mount('/content/drive') # Remount to perform Git operations

os.chdir(GIT_REPO_PATH)

try:
    subprocess.run(["git", "add", "."], check=True)
    subprocess.run(["git", "commit", "-m", "Automated commit: completed AnySat AGB pipeline run"], check=True)
    print(f"Pushing updates to branch: {BRANCH_NAME}...")
    subprocess.run(["git", "push", "origin", BRANCH_NAME], check=True)
    print("Code successfully committed and pushed to GitHub.")
except subprocess.CalledProcessError as e:
    print(f"Git operation failed. There might be no changes to commit, or a merge conflict occurred: {e}")

In [ ]:
# Gracefully terminate the Colab Runtime to save compute credits
from google.colab import runtime
print("Pipeline complete. All data safely stored. Disconnecting runtime...")
runtime.unassign()